In [125]:
import pandas as pd
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Forme :", df.shape)
print(df.dtypes)
df.head()

Forme : (7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [126]:
def audit_qualite(df):
    print("\nForme (lignes, colonnes) :", df.shape)
    manquants = df.isna().sum()
    # print(manquants)
    pourcentage = (df.isna().mean() * 100).round(1)
    # print(pourcentage)
    resume = pd.DataFrame({"manquants": manquants, "pourcent": pourcentage}).sort_values("pourcent", ascending=False)
    if(len(resume[resume["manquants"] > 0])==0):
        print("Manquants détectés : 0 colonne")
    else:
        print(resume[resume["manquants"] > 0].sort_values("pourcent", ascending=False))
    
    total = len(df)
    if total == 0:
        print("Dataset vide")
        return
    churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"], fill_value=0)
    churn_pct = (churn_counts / total * 100).round(1)

    print(f"Cible Churn : No={churn_counts['No']} ({churn_pct['No']}%), Yes={churn_counts['Yes']} ({churn_pct['Yes']}%)")

    if churn_pct.max() > 70:
        print("Attention : cible déséquilibrée")
    


In [127]:
#tous le dataset
audit_qualite(df)


Forme (lignes, colonnes) : (7043, 21)
Manquants détectés : 0 colonne
Cible Churn : No=5174 (73.5%), Yes=1869 (26.5%)
Attention : cible déséquilibrée


In [128]:
#dataset filtré churn yes
df_filtered = df[df["Churn"] == "Yes"]
audit_qualite(df_filtered)



Forme (lignes, colonnes) : (1869, 21)
Manquants détectés : 0 colonne
Cible Churn : No=0 (0.0%), Yes=1869 (100.0%)
Attention : cible déséquilibrée


In [129]:
#dataset filtré churn no
df_filtered = df[df["Churn"] == "No"]
audit_qualite(df_filtered)



Forme (lignes, colonnes) : (5174, 21)
Manquants détectés : 0 colonne
Cible Churn : No=5174 (100.0%), Yes=0 (0.0%)
Attention : cible déséquilibrée


In [130]:
def reparer_total_charges(df):
    
    import numpy as np
    s = df["TotalCharges"].astype(str).str.strip()

    # 1) colonne 100% non numérique -> on refuse
    converted = pd.to_numeric(s, errors="coerce")
    if converted.notna().sum() == 0:
        raise ValueError("TotalCharges est entièrement non numérique, conversion refusée")

    # 2) repérer les valeurs suspectes type 29,90
    suspects = s.str.match(r"^\d+,\d+$", na=False)
    if suspects.any():
        exemples = df.loc[suspects, "TotalCharges"].head(5).tolist()
        raise ValueError(f"Format numérique invalide détecté dans TotalCharges : {exemples}")
    
    totalCharges = df["TotalCharges"]
    totalChargesNum = pd.to_numeric(df["TotalCharges"], errors="coerce")
    print("Trous avant :", int(totalCharges.isna().sum()))
    print("Trous après conversion :", int(np.isnan(totalChargesNum).sum()))
    print(f"Pourcentage : {int(np.isnan(totalChargesNum).sum())/len(totalChargesNum)*100:.2f}")

    print("Imputation simple")
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy="median")
    df1 = pd.DataFrame({"TotalCharges": totalChargesNum})
    totalCharge_rempli = imputer.fit_transform(df1[["TotalCharges"]])
    print("Valeur utilisée (médiane) :", round(imputer.statistics_[0], 1))
    print(f"Avant : moyenne={df1['TotalCharges'].mean():.2f}, écart-type={df1['TotalCharges'].std():.2f}")
    print(f"Après : moyenne={pd.Series(totalCharge_rempli.ravel()).mean():.2f}, écart-type={pd.Series(totalCharge_rempli.ravel()).std():.2f}")
    print("Suppression ligne vide")
    df_sans_lignes = pd.DataFrame({"TotalCharges": totalChargesNum}).dropna(subset=["TotalCharges"])
    print(f"Avant : moyenne={df1['TotalCharges'].mean():.2f}, écart-type={df1['TotalCharges'].std():.2f}")
    print(f"Après : moyenne={pd.Series(df_sans_lignes['TotalCharges']).mean():.2f}, écart-type={pd.Series(df_sans_lignes['TotalCharges']).std():.2f}")

    if (int(np.isnan(totalChargesNum).sum())/len(totalChargesNum)*100 < 1):
        df["TotalCharges"] = df_sans_lignes["TotalCharges"]
    else:
        df["TotalCharges"] = totalCharge_rempli.ravel()

    return df


In [131]:
df = reparer_total_charges(df)

Trous avant : 0
Trous après conversion : 11
Pourcentage : 0.16
Imputation simple
Valeur utilisée (médiane) : 1397.5
Avant : moyenne=2283.30, écart-type=2266.77
Après : moyenne=2281.92, écart-type=2265.27
Suppression ligne vide
Avant : moyenne=2283.30, écart-type=2266.77
Après : moyenne=2283.30, écart-type=2266.77


In [132]:
def encoder_features(df):
    # Encode all categorical columns.
    # Return a 100% numeric df, ready for a model.
    # TODO : rep?rer les colonnes cat?gorielles (dtype object)
    # TODO : pour les binaires Yes/No, un encodage simple 0/1
    # TODO : pour les nominales (PaymentMethod, Contract...), du One-Hot
    # TODO : d?cider du sort de customerID (indice : un identifiant n'est PAS une feature)
    # pass
    colonnes_categorielles = df.select_dtypes(include=["object", "string"]).columns.tolist()
    print(colonnes_categorielles)
    if "customerID" in colonnes_categorielles:
        colonnes_categorielles.remove("customerID")

    colonnes_binaires = [col for col in colonnes_categorielles if df[col].nunique(dropna=True) == 2]
    colonnes_nominales = [col for col in colonnes_categorielles if col not in colonnes_binaires]

    print("Colonnes binaires :", colonnes_binaires)
    print("Colonnes nominales :", colonnes_nominales)

    for col in colonnes_binaires:
        valeurs = df[col].dropna().unique().tolist()
        if set(valeurs) == {"Yes", "No"} or set(valeurs) == {"No", "Yes"}:
            df[col] = df[col].map({"No": 0, "Yes": 1})
        else:
            df[col] = pd.Categorical(df[col]).codes

    if colonnes_nominales:
        df = pd.get_dummies(df, columns=colonnes_nominales, prefix=colonnes_nominales, dtype=int)
    ids = df["customerID"]
    df = df.drop(columns=["customerID"])
    print(df.shape)
    return df


In [133]:
df = encoder_features(df)

['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
Colonnes binaires : ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
Colonnes nominales : ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
(7043, 41)
